### <div style="background-color:blue; color:white; padding:10px;"> Imports </div>

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

### <div style="background-color:blue; color:white; padding:10px;"> Paths and parameters </div>

In [39]:
# Dataset paths
RGB_ROOT = r"D:\Datasets\Datasets\ADL\Frames\P_16\Original_P_16"
FLOW_ROOT = r"D:\Datasets\Datasets\ADL\OpticalFlow\P_16\viz"
ANNOTATION_CSV = r"D:\Datasets\Datasets\ADL\Label\P_16_labeled.csv"
# Output paths
OUTPUT_FEATURES_CSV = "../Features/ADL/P_16_fused_features.csv"
OUTPUT_LABELS_CSV = "../Labels/ADL/P_16_fused_labeled.csv"
# Model parameters
FEATURE_DIM = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


### <div style="background-color:blue; color:white; padding:10px;">Load annotation CSV</div>

In [40]:
annotations = pd.read_csv(ANNOTATION_CSV)
print("Total annotations:", len(annotations))
annotations.head()

Total annotations: 17


,StartFrame,EndFrame,ActionLabel,ActionName
0,420,2760,3,brushing teeth
1,2850,3090,6,drying hands/face
2,3150,3720,1,combing hair
3,3750,4500,5,washing hands/face
4,4530,4830,6,drying hands/face


### <div style="background-color:blue; color:white; padding:10px;"> Create frame-level multi-label matrix </div>

In [41]:
max_frame = annotations['EndFrame'].max()
num_classes = annotations['ActionLabel'].nunique()
print("Total frames:", max_frame)
print("Total classes:", num_classes)

Total frames: 25200
Total classes: 14


In [42]:
# labels = np.zeros((max_frame + 1, num_classes), dtype=np.float32)
# for _, row in annotations.iterrows():    
#     start = int(row['StartFrame'])
#     end = int(row['EndFrame'])
#     cls = int(row['ActionLabel'])    
#     labels[start:end+1, cls] = 1.0
# print("Labels shape:", labels.shape)

# Get unique labels and create a mapping to contiguous 0-based indices
unique_labels = sorted(annotations['ActionLabel'].unique())
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
num_classes = len(unique_labels)

print("Unique ActionLabels:", unique_labels)
print("Label → Index mapping:", label_to_idx)

max_frame = annotations['EndFrame'].max()
labels = np.zeros((max_frame + 1, num_classes), dtype=np.float32)

for _, row in annotations.iterrows():
    start = int(row['StartFrame'])
    end = int(row['EndFrame'])
    cls = label_to_idx[int(row['ActionLabel'])]  # map to 0-based index
    labels[start:end + 1, cls] = 1.0

print("Labels shape:", labels.shape)

Unique ActionLabels: [1, 3, 5, 6, 8, 9, 10, 12, 14, 15, 16, 18, 20, 22]
Label → Index mapping: {1: 0, 3: 1, 5: 2, 6: 3, 8: 4, 9: 5, 10: 6, 12: 7, 14: 8, 15: 9, 16: 10, 18: 11, 20: 12, 22: 13}
Labels shape: (25201, 14)


### <div style="background-color:blue; color:white; padding:10px;"> Define image transform </div>

In [43]:
transform = transforms.Compose([    
    transforms.Resize((224, 224)),    
    transforms.ToTensor(),    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### <div style="background-color:blue; color:white; padding:10px;">Load ResNet-50 feature extractor </div>

In [44]:
resnet = models.resnet50(weights=True)
# Remove classifier
modules = list(resnet.children())[:-1]
resnet = nn.Sequential(*modules)
# Add projection layer
projection = nn.Linear(2048, FEATURE_DIM)
# projection = nn.Identity()
# FEATURE_DIM = 2048
resnet = resnet.to(DEVICE)
projection = projection.to(DEVICE)
resnet.eval()
projection.eval()

C:\Users\PAWANESH\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Linear(in_features=2048, out_features=512, bias=True)

In [45]:
@torch.no_grad()
def extract_feature(image_path):    
    image = Image.open(image_path).convert("RGB")    
    image = transform(image).unsqueeze(0).to(DEVICE)    
    feature = resnet(image)    
    feature = feature.view(1, -1)    
    feature = projection(feature)    
    feature = feature.squeeze(0).cpu().numpy()    
    return feature

### <div style="background-color:blue; color:white; padding:10px;"> Load all frame paths</div>

In [46]:
rgb_frames = sorted(glob.glob(os.path.join(RGB_ROOT, "*.jpg")))
flow_frames = sorted(glob.glob(os.path.join(FLOW_ROOT, "*.jpg")))
assert len(rgb_frames) == len(flow_frames), "Mismatch in RGB and Flow frames"
num_frames = len(rgb_frames)
print("Total frames:", num_frames)

Total frames: 25197


### <div style="background-color:blue; color:white; padding:10px;"> Extract RGB and Flow features </div>

In [ ]:
rgb_features = np.zeros((num_frames, FEATURE_DIM), dtype=np.float32)
flow_features = np.zeros((num_frames, FEATURE_DIM), dtype=np.float32)
for i in tqdm(range(num_frames), desc="Extracting features"):    
    rgb_features[i] = extract_feature(rgb_frames[i])    
    flow_features[i] = extract_feature(flow_frames[i])
print("RGB features:", rgb_features.shape)
print("Flow features:", flow_features.shape)

Extracting features:  25%|█████████████▊                                          | 6192/25197 [07:36<20:09, 15.71it/s]

### <div style="background-color:blue; color:white; padding:10px;"> Adaptive multimodal fusion </div>

We implement adaptive fusion layer:

$$f_t=\alpha {f_t}^{rgb} + (1-\alpha) {f_t}^{flow}$$

In [ ]:
class AdaptiveFusion(nn.Module):    
    def __init__(self, dim):        
        super().__init__()        
        self.fc = nn.Linear(dim * 2, 1)       
        self.sigmoid = nn.Sigmoid()   
    def forward(self, rgb, flow):        
        x = torch.cat([rgb, flow], dim=1)        
        alpha = self.sigmoid(self.fc(x))        
        fused = alpha * rgb + (1 - alpha) * flow        
        return fused

In [ ]:
# Initialize fusion:
fusion_model = AdaptiveFusion(FEATURE_DIM).to(DEVICE)
fusion_model.eval()

In [ ]:
# Apply fusion:
rgb_tensor = torch.from_numpy(rgb_features).to(DEVICE)
flow_tensor = torch.from_numpy(flow_features).to(DEVICE)
with torch.no_grad():   
    fused_tensor = fusion_model(rgb_tensor, flow_tensor)
fused_features = fused_tensor.cpu().numpy()
print("Fused features shape:", fused_features.shape)

### <div style="background-color:blue; color:white; padding:10px;">Save fused features and labels to CSV </div>

In [ ]:
features_df = pd.DataFrame(fused_features)
features_df.insert(0, "frame_id", np.arange(num_frames))
features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)
print("Saved fused features:", OUTPUT_FEATURES_CSV)

In [ ]:
# Correct alignment fix
num_frames = len(rgb_frames)
label_frames = labels.shape[0]
if label_frames < num_frames:    
    padding = np.zeros((num_frames - label_frames, labels.shape[1]), dtype=np.float32) 
    labels = np.vstack([labels, padding])
elif label_frames > num_frames:
    labels = labels[:num_frames]
print("Final labels shape:", labels.shape)
labels_df = pd.DataFrame(labels)
labels_df.insert(0, "frame_id", np.arange(num_frames))
labels_df.to_csv(OUTPUT_LABELS_CSV, index=False)
print("Saved labels:", OUTPUT_LABELS_CSV)

In [ ]:
# Save mapping for later use (e.g., during inference/evaluation)
# label_mapping_df = pd.DataFrame({
#     'ActionLabel': list(label_to_idx.keys()),
#     'MappedIndex': list(label_to_idx.values())
# })
# label_mapping_df.to_csv("../Labels/ADL/P_11_label_mapping.csv", index=False)